In [0]:
from pyspark.sql.functions import (
    current_timestamp, lit, col, to_date,
    sum as spark_sum, current_date, to_timestamp
)
from pyspark.sql.window import Window

VOLUME_PATH = "/Volumes/de_workspace26/ecommerce_pawan/raw_files"
CATALOG     = "de_workspace26"
SCHEMA_B    = f"{CATALOG}.bronze_pawan"
SCHEMA_S    = f"{CATALOG}.silver_pawan"
SCHEMA_G    = f"{CATALOG}.gold_pawan"

print("Constants set.")
print("Volume path :", VOLUME_PATH)

In [0]:
bronze_orders  = spark.read.table(f"{SCHEMA_B}.orders")
deduped_orders = bronze_orders.dropDuplicates(["order_id"])

print(f"Before dedup : {bronze_orders.count()}")   # 205
print(f"After  dedup : {deduped_orders.count()}")  # 200
# ORD0001, ORD0002, ORD0003, ORD0004, ORD0005 appear twice → 5 removed

In [0]:
orders_typed = (deduped_orders
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
    .withColumn("revenue", col("quantity") * col("unit_price"))
)

orders_typed.select(
    "order_id", "order_date", "quantity", "unit_price", "revenue"
).show(5)

In [0]:
window_spec = (Window
    .partitionBy("customer_id")
    .orderBy("order_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

orders_windowed = orders_typed.withColumn(
    "cumulative_revenue",
    spark_sum("revenue").over(window_spec)
)

orders_windowed.select(
    "order_id", "customer_id", "order_date",
    "revenue", "cumulative_revenue"
).show(10)

In [0]:
customers_bronze = spark.read.table(f"{SCHEMA_B}.customers")
products_bronze  = spark.read.table(f"{SCHEMA_B}.products")

enriched_orders = (orders_windowed
    .join(
        customers_bronze.select("customer_id", "city", "loyalty_tier"),
        on="customer_id", how="left"
    )
    .join(
        products_bronze.select("product_id", "product_name", "category"),
        on="product_id", how="left"
    )
)

print("Enriched orders row count:", enriched_orders.count())  # 200

enriched_orders.select(
    "order_id", "customer_id", "city", "loyalty_tier",
    "product_name", "category", "revenue"
).show(5)

In [0]:
(enriched_orders.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("region")
    .saveAsTable(f"{SCHEMA_S}.orders")
)

count = spark.read.table(f"{SCHEMA_S}.orders").count()
print(f"✅ silver_pawan.orders written — {count} rows")  # 200
print("   Partitions: North, South, East, West")

# Verify partition-wise counts
spark.sql(f"""
    SELECT region, COUNT(*) AS row_count
    FROM   {SCHEMA_S}.orders
    GROUP BY region
    ORDER BY region
""").show()

In [0]:
customers_bronze = spark.read.table(f"{SCHEMA_B}.customers")

scd_init = (customers_bronze
    .withColumn("is_current",           lit(True))
    .withColumn("effective_start_date", current_date())
    .withColumn("effective_end_date",   lit(None).cast("date"))
)

scd_init.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{SCHEMA_S}.customers")

print(f"✅ silver_pawan.customers initial load done")
print(f"   Row count : {spark.read.table(f'{SCHEMA_S}.customers').count()}")  # 20
print(f"   All is_current = True")

spark.read.table(f"{SCHEMA_S}.customers") \
     .select("customer_id", "name", "loyalty_tier",
             "is_current", "effective_start_date", "effective_end_date") \
     .show(5)

In [0]:
customers_update = spark.read.csv(
    f"{VOLUME_PATH}/customers_update.csv",
    header=True, inferSchema=True
)
customers_update.createOrReplaceTempView("customers_update_view")

print("customers_update rows:", customers_update.count())  # 10 (CUST01–CUST10)
customers_update.show()

In [0]:
# Close existing current records where loyalty_tier has changed
spark.sql(f"""
    MERGE INTO {SCHEMA_S}.customers AS target
    USING customers_update_view     AS source
    ON  target.customer_id = source.customer_id
    AND target.is_current  = true
    WHEN MATCHED AND target.loyalty_tier <> source.loyalty_tier THEN
      UPDATE SET
        target.is_current         = false,
        target.effective_end_date = current_date()
""")

print("✅ Pass 1 done — old records closed.")

# Verify closed records
spark.sql(f"""
    SELECT customer_id, loyalty_tier, is_current,
           effective_start_date, effective_end_date
    FROM   {SCHEMA_S}.customers
    WHERE  is_current = false
    ORDER BY customer_id
""").show()
# Expected: 10 rows closed (CUST01–CUST10 all had loyalty_tier changes)

In [0]:
spark.sql(f"""
    MERGE INTO {SCHEMA_S}.customers AS target
    USING (
        SELECT s.*
        FROM   customers_update_view s
        JOIN   {SCHEMA_S}.customers  t
          ON   s.customer_id        = t.customer_id
         AND   t.is_current         = false
         AND   t.effective_end_date = current_date()
    ) AS source
    ON target.customer_id = source.customer_id
    AND target.is_current = true
    WHEN NOT MATCHED THEN
      INSERT (customer_id, name, email, city, loyalty_tier, signup_date,
              is_current, effective_start_date, effective_end_date)
      VALUES (source.customer_id, source.name, source.email, source.city,
              source.loyalty_tier, source.signup_date,
              true, current_date(), null)
""")

print("✅ Pass 2 done — new current records inserted.")
total = spark.read.table(f"{SCHEMA_S}.customers").count()
print(f"   Total rows in silver_pawan.customers : {total}")
# Expected: 30 (20 original + 10 new records for CUST01–CUST10)

In [0]:
# Show both old and new records for changed customers
spark.sql(f"""
    SELECT customer_id, loyalty_tier, is_current,
           effective_start_date, effective_end_date
    FROM   {SCHEMA_S}.customers
    WHERE  customer_id IN ('CUST01','CUST02','CUST03',
                           'CUST04','CUST05','CUST06')
    ORDER BY customer_id, effective_start_date
""").show()

# Expected per customer:
#   CUST01: Gold(is_current=false)   + Bronze(is_current=true)
#   CUST02: Bronze(is_current=false) + Silver(is_current=true)
#   CUST03: Bronze(is_current=false) + Gold(is_current=true)
#   CUST04: Bronze(is_current=false) + Silver(is_current=true)
#   CUST05: Gold(is_current=false)   + Bronze(is_current=true)
#   CUST06: Bronze(is_current=false) + Gold(is_current=true)

In [0]:
spark.sql(f"""
    ALTER TABLE {SCHEMA_S}.orders
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
print("✅ Change Data Feed enabled on silver_pawan.orders")

# Trigger an update to generate CDF records
spark.sql(f"""
    UPDATE {SCHEMA_S}.orders
    SET    status = 'delivered'
    WHERE  status = 'shipped' AND region = 'North'
""")
print("✅ Update applied — shipped → delivered for North region")

In [0]:
spark.sql(f"""
    SELECT _change_type, order_id, status, region, _commit_version
    FROM   table_changes('{SCHEMA_S}.orders', 1)
    ORDER BY _commit_version
    LIMIT  15
""").show(truncate=False)

# _change_type values:
#   update_preimage  → row before update
#   update_postimage → row after update

In [0]:
spark.sql(f"""
    OPTIMIZE {SCHEMA_S}.orders
    ZORDER BY (customer_id, order_date)
""")
print("✅ OPTIMIZE + Z-ORDER complete on silver_pawan.orders")
print("   Co-located data for fast customer_id + order_date filters.")

In [0]:
print("=" * 50)
print("SILVER LAYER SUMMARY")
print("=" * 50)

silver_orders    = spark.read.table(f"{SCHEMA_S}.orders")
silver_customers = spark.read.table(f"{SCHEMA_S}.customers")

print(f"silver_pawan.orders    rows : {silver_orders.count()}")     # 200
print(f"silver_pawan.customers rows : {silver_customers.count()}")  # 30

print("\nOrders schema:")
silver_orders.printSchema()

print("\nCustomers schema:")
silver_customers.printSchema()

print("\nOrders partition counts:")
spark.sql(f"""
    SELECT region, COUNT(*) AS row_count
    FROM   {SCHEMA_S}.orders
    GROUP BY region ORDER BY region
""").show()

print("\nCustomer SCD status:")
spark.sql(f"""
    SELECT is_current, COUNT(*) AS count
    FROM   {SCHEMA_S}.customers
    GROUP BY is_current
""").show()
# is_current=true  → 20 (10 unchanged + 10 new)
# is_current=false → 10 (old records for CUST01–CUST10)